# Mythos in ArCo

## Introduction

This project is an **exploratory investigation of the ArCo knowledge graph**, with a specific focus on historical and artistic cultural heritage assets related to **mythology**.

The objective of the project is to identify and analyze mythological subjects represented within cultural heritage objects preserved in ArCo, exploring how mythological figures, creatures, and narratives appear across different historical periods, artistic contexts, and cultural institutions.

Since ArCo does not provide a dedicated controlled vocabulary for mythology-related subjects, <b>Wikidata</b> was used as an external Linked Open Data source to enrich the dataset semantically.
A mythology-related reference dataset was therefore created by extracting entities from Wikidata associated with mythological figures, deities, legendary characters, and other mythology-related categories.

The resulting dataset was then used as a semantic reference vocabulary to filter and identify ArCo items whose subjects are connected to mythology.

## Acquiring and parsing data

### Preliminary study

The very first step was to study ArCo and Wikidata datasets. For starters, the **existing alignment** between Arco historical-artistic resources and Wikidata entities was investigated, in order to understand if works recorded in Arco were described in Wikidata as well, and to eventually retrieve the associated mythological character or episod.

In [8]:
import SPARQLWrapper
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

In [9]:
#ENDPOINT SPARLQ DI ARCO 
arco_endpoint = "https://dati.cultura.gov.it/sparql"

sparql_arco = SPARQLWrapper(arco_endpoint)
sparql_arco.setReturnFormat(JSON)

In [10]:
query_arco_same_as = """
    SELECT ?culturalProperty ?wikidataEntity ?label
    WHERE { 
        ?culturalProperty rdf:type arco:HistoricOrArtisticProperty ;
                          owl:sameAs ?wikidataEntity .
        FILTER(STRSTARTS(STR(?wikidataEntity), "http://www.wikidata.org/entity/"))

    # etichetta del bene in Arco se c'è
    OPTIONAL { ?culturalProperty rdfs:label ?label }
    FILTER(LANG(?label) = "en")
    }
"""

sparql_arco.setQuery(query_arco_same_as)
results_same_as = sparql_arco.query().convert()

df_results = pd.DataFrame(results_same_as['results']['bindings'])
df_same_as = df_results.map(lambda x: x['value'] if isinstance(x, dict) and 'value' in x else x)

df_same_as
#len(df_same_as)

,culturalProperty,wikidataEntity,label
0,https://w3id.org/arco/resource/HistoricOrArtis...,http://www.wikidata.org/entity/Q546297,"Cristo morto nel sepolcro e tre dolenti, compi..."
1,https://w3id.org/arco/resource/HistoricOrArtis...,http://www.wikidata.org/entity/Q2415504,David con la testa di Golia (statua) by Cioni ...
2,https://w3id.org/arco/resource/HistoricOrArtis...,http://www.wikidata.org/entity/Q2627028,ritratto di Elisabetta Gonzaga (dipinto) by Ra...
3,https://w3id.org/arco/resource/HistoricOrArtis...,http://www.wikidata.org/entity/Q2344505,"Madonna del cardellino, Madonna con Bambino e ..."
4,https://w3id.org/arco/resource/HistoricOrArtis...,http://www.wikidata.org/entity/Q3842642,"Madonna del solletico, Madonna con Bambino (di..."


Reconciliation was also investigated starting from Wikidata dataset, looking for entities provided with ICCD identifiers or with identifiers from Catalogo Generale dei Beni Culturali, from which ArCo Knowledge Graph pulls its data.

#Stessa ricerca da wikidata. esempio di descrizione opera wikidata (per estrarre tutti i mythological paitings ed entità associate e seguire rete di relazioni); esempio di query sia da arco sia da wikidata (uso iccd e codice catalogo beni culturali, da cui arco prende i dati); conclusioni e collegamento con parte su wikidata

In [11]:
#richiamo l'endpoint di wikidata 
wikidata_endpoint = "https://query.wikidata.org/sparql"
sparql_wd = SPARQLWrapper(wikidata_endpoint)

In [7]:
query_wd_arco_id = """
    SELECT ?item ?arcoId 
    WHERE {
        ?item wdt:P9051 ?arcoId .
        #filtra solo proprietà storico artistiche
        FILTER(STRSTARTS(STR(?arcoId), "HistoricOrArtisticProperty"))
        # estrai solo opere mitologiche
        ?item wdt:P136 ?myth .
        ?myth wdt:P279 wd:Q115574326 .
    }
"""

""" sparql_wd.setQuery(query_wd_arco_id)
results_wd_arco_id = sparql_wd.query().convert() """

""" df_results_wd = pd.DataFrame(results_wd_arco_id['results']['bindings'])
df_wd_arco_id = df_results_wd.map(lambda x: x['value'] if isinstance(x, dict) and 'value' in x else x)

df_wd_arco_id """

" df_results_wd = pd.DataFrame(results_wd_arco_id['results']['bindings'])\ndf_wd_arco_id = df_results_wd.map(lambda x: x['value'] if isinstance(x, dict) and 'value' in x else x)\n\ndf_wd_arco_id "

### Wikidata dataset

#### Collecting mythological entities from Wikidata

### ArCo dataset

In [12]:
import requests
import time 
from tqdm import tqdm

In [13]:
#carichiamo il csv con i soggetti di Wikidata
df_wikidata = pd.read_csv("wd_dataset_pulito.csv", encoding="utf-8")

df_wikidata.head(3)


,entity,label,aliases,description,category
0,http://www.wikidata.org/entity/Q279782,Abarbarea,NaN,naiade della mitologia greca,greek_deity
1,http://www.wikidata.org/entity/Q2370052,Acanto,NaN,"personaggio della mitologia greca, probabilmen...",greek_deity
2,http://www.wikidata.org/entity/Q4668463,Acaste,NaN,Oceanina nella mitologia greca,greek_deity


After creating the Wikidata mythology dataset, the next step focused on preparing a reference vocabulary that could be used to identify mythology-related subjects within the ArCo dataset.

The filtering process was based on the **labels and aliases** extracted from Wikidata entities. 
These terms constituted the vocabulary used to match mythological subjects appearing in ArCo metadata.

In [14]:
#estraggo tutte le label di wikidata 
labels = (
    df_wikidata["label"]
    .dropna() #tolgo i valori nulli
    .str.strip() #tolgo gli spazi bianchi all'inizio e alla fine
    .tolist() #converto in lista

)

#estraggo gli aliases di wikidata separando quelli multipli
aliases = []
for alias_str in df_wikidata["aliases"].dropna(): #per ogni stringa di alias che non è nulla
    alias_list = [alias.strip() for alias in alias_str.split(",")] #separo gli alias con la virgola, tolgo gli spazi e metto tutto in minuscolo
    aliases.extend(alias_list) #aggiungo alla lista totale degli alias

#unisco tutto togliendo i duplicati
myth_terms = list(set(labels + aliases)) 
len(myth_terms)

2928

The next step focused on accessing the ArCo knowledge graph in order to identify cultural heritage items related to mythology.

The ArCo SPARQL endpoint was accessed through the SPARQLWrapper Python library, allowing direct querying of the knowledge graph.

In [15]:
#ENDPOINT SPARLQ DI ARCO 
arco_endpoint = "https://dati.cultura.gov.it/sparql"

sparql_arco = SPARQLWrapper(arco_endpoint)
sparql_arco.setReturnFormat(JSON)

Before applying mythology-related filters, an initial exploratory query was performed to determine the **total number of historical and artistic cultural heritage objects** available in ArCo.

The query targeted entities classified as HistoricOrArtisticProperty, which represent historical and artistic cultural heritage assets within the ArCo ontology.

In [16]:
# NUMERO TOTALE DEI BENI STORICO ARTISTICI - senza duplicati 
query1 = """
PREFIX arco: <https://w3id.org/arco/ontology/arco/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 

SELECT (COUNT(DISTINCT ?item) AS ?totalItems)
WHERE {
  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
}
"""


sparql_arco.setQuery(query1)
results = sparql_arco.query().convert()

for result in results["results"]["bindings"]:
    print(f"Total Historic or Artistic Properties: {result['totalItems']['value']}")

Total Historic or Artistic Properties: 2259266


For each cultural heritage item, the extraction process retrieved:

- the unique item URI,
- the item label,
- and the associated subject information.

The subject field was particularly important because it represents the semantic description of the cultural object and constitutes the main element used to identify mythology-related items.

Due to the large size of the ArCo knowledge graph, the extraction process was performed incrementally using **batch queries**.

Instead of retrieving the entire dataset in a single request, the workflow divided the extraction into smaller portions of data using LIMIT and OFFSET parameters within the SPARQL query.

Each iteration retrieved a subset of cultural heritage items and progressively appended the results to a cumulative list.

In [17]:
#query pr estrarre subject e label dei beni storico artistici in batch (altrimenti esplode)
# Iterare su tutti i 2.258.640 beni con batch di 20000 (altrimenti esplode)

rows = []
batch_size = 20000
total_items = 2258640

for offset in range(0, total_items, batch_size):
    print(f"Processando batch: offset={offset}, items processati={len(rows)}")
    
    query2 = f"""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX a-cd: <https://w3id.org/arco/ontology/context-description/>

SELECT ?item ?subject ?label
WHERE {{

  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
  
  OPTIONAL {{ ?item rdfs:label ?label . }}

  OPTIONAL {{ ?item a-cd:subject ?subject . }}

}}
LIMIT {batch_size}
OFFSET {offset}
"""
    
    sparql_arco.setQuery(query2)
    results = sparql_arco.query().convert()
    
    # estrai i dati da questo batch
    for r in results["results"]["bindings"]:
        rows.append({
            "item": r["item"]["value"] if "item" in r else None,
            "label": r["label"]["value"] if "label" in r else None,
            "subject": r["subject"]["value"] if "subject" in r else None
        })
    
    # se il batch è vuoto, ferma il loop
    if len(results["results"]["bindings"]) < batch_size:
        print(f"Batch incompleto ({len(results['results']['bindings'])} items), fine loop")
        break


Processando batch: offset=0, items processati=0
Processando batch: offset=20000, items processati=20000
Processando batch: offset=40000, items processati=40000
Processando batch: offset=60000, items processati=60000
Processando batch: offset=80000, items processati=80000
Processando batch: offset=100000, items processati=100000
Processando batch: offset=120000, items processati=120000
Processando batch: offset=140000, items processati=140000
Processando batch: offset=160000, items processati=160000
Processando batch: offset=180000, items processati=180000
Processando batch: offset=200000, items processati=200000
Processando batch: offset=220000, items processati=220000
Processando batch: offset=240000, items processati=240000
Processando batch: offset=260000, items processati=260000
Processando batch: offset=280000, items processati=280000
Processando batch: offset=300000, items processati=300000
Processando batch: offset=320000, items processati=320000
Processando batch: offset=340000

After completing all extraction batches, the retrieved data was consolidated into a single DataFrame.

In [18]:

# creiamo il dataframe consolidato
df_arco = pd.DataFrame(rows)
df_arco = df_arco.drop_duplicates(subset="item")

print(f"Righe totali in df_arco : {len(df_arco)}")
df_arco[["item", "label", "subject"]].head(1)

Righe totali in df_arco : 1120196


,item,label,subject
0,https://w3id.org/arco/resource/HistoricOrArtis...,Arlecchino primordiale. Fine del cinquecento. ...,"figurino per costume di scena, maschile"


## Filtering and cleaning data

After extracting all historical-artistic resources with subject information from ArCo, data was filtered in order to retrieve only the works which represented mythological characters and events described in Wikidata. 

The filtering process was based on <b>textual matching</b> between the ArCo subject field and the mythology vocabulary created from Wikidata labels and aliases. 

As a preliminary step, all cultural heritage items without subject metadata were removed from the dataset. Since the identification of mythology-related works relied entirely on subject descriptions, records lacking subject information could not contribute to the analysis.

The vocabulary was transformed into a regex-compatible list of terms. We excluded null values, empty strings and non-capitalized terms. This decision helped reduce noise and false positives such as "atlante" or "flora".

A major challenge of the filtering process concerned ambiguous terms that may refer either to mythological entities or to unrelated concepts. For this reason, two complementary validation systems were introduced: contextual inclusion rules and esclusion rules.
- The <b>context rules</b> accepts mythological names only if they appear together with specific contextual keywords associated with mythology.
- The <b>exclusion rules</b> excludes terms appearing in contexts not related to mythology. 

After those steps, only the items containing at least one validated mythology-related match were kept. 

In [19]:
#ESTRAIAMO SOLO I SOGGETTI MITOLOGICI 

import re

# 1. filtro base --> togliamo i NaN
df_arco = df_arco[df_arco["subject"].notna()].copy()

#creiamo la nostra lista di termini myth_terms
terms = [
    re.escape(t)
    for t in myth_terms
    if isinstance(t, str)
    and t.strip() != ""
    and t[0].isupper()  
]

# regex unica case-sensitive e per evitare termini composti
pattern = r'(?<![A-Za-zÀ-ÿ])(' + '|'.join(terms) + r')(?![A-Za-zÀ-ÿ])'
regex = re.compile(pattern)


#TERMINI DA ACCETTARE SOLO SE COMPAIONO INSIEME A QUESTE PAROLE []
context_rules = {
    "Ascanio" : ["fuoco", "cervo"],"Callisto": ["Arcade", "Diana"],"Concordia": ["Allegoria", "Innocenza", "Giustizia"],
    "Cornacchia": ["metamorfosi"],"Egeo": ["Teseo"],"Elena": ["ratto"],"Ettore": ["morte", "Protesilao", "Achille", "Andromaca", "Paride"],
    "Europa": ["ratto", "Toro", "sibilla", "mito","allegoria","ancelle","leone"],
    "Grazie" : ["tre", "danzanti"],"Ida" : ["Linceo"],"Ippolito":["episodi", "caduta", "morte"],
    "Nilo":["personificazione","allegoria"],"Roma":["dea"],"Silvano":["Dio"],"Sonno":["allegoria"],
    "Vesta":["Giove"],"Vittoria":["allegoria"]
}


# TERMINI DA ESCLUDERE SE COMPAIONO CON DETERMINATE PAROLE
exclusion_rules = {
    "Abbondanza": ["Santa", "Sant","san","papa"],"Amore": ["Dio", "Madonna", "castello", "fontana","antica"],"Andromeda":["galassia"],"Apollo":["sala","tempio"],
    "Atlante":["Scienze","turisti"],"Carità": ["Santa", "Sant", "Maria"],
    "Carita": ["Santa", "Sant", "Maria"],"Diana" : ["Beata", "duca","scenografia","tempio","conte","san"],
    "Dioniso": ["stemma","stemmi", "san", "santi", "s."],"Dionisio":["Ritratto","san","barletta","beati","vescovo","Santi","stemma","papa","padre","beato"],
    "Elio" : ["ponte","ritratto","paesaggio", "veduta"],"Enea": ["Beato", "discepolo","stemma", "papa", "silvio"],
    "Ercole": ["Ricotti", "Dandini", "Comini", "Farnese","cardinale", "tempio", "stemma", "ritratto", "Roccotti", "colonne", "porto", "basilica", "santa"],
    "Ermes" : ["ritratto"],"Esculapio": ["tempio"],"Satiro":["san","santi","santo","fratello"],
    "Fede" : ["santi", "sant", "san", "dottrina", "biblici", "Francia", "cattolica", "madonna", "evangelisti", "religione", "dio", "angeli", "angioletti","santa", "papa", "santo", "cristiana", "chiesa", "miracolo", "misericordia","virtù", "ritratto", "stemma"],
    "Flora" :["ritratto", "sante", "circo", "ruota", "iside", "madonna", "tempio"],"Galatea":["ritratto"],
    "Giasone":["san","ritratto"],"Giove":["san","tempio"],"Musa":["tempio"],
    "Giunone":["scenografia", "tempio"],"Giustizia" : ["porta", "palazzo", "misericordia", "san", "santa", "papa", "imperatore", "papale","Virtù", "apostoli","papa","Giglio"],
    "Gorgone" : ["santa"],"greco":["testo","teatro"],"Io": ["ritratti "],"Ippolito":["Nievo"],"Lavinia":["ritratto", "autoritratto"],
    "Leandro" : ["autoritratto","ritratto","stemma","San", "busto", "torre"],"Marte":["tempio","sala","ultore"],
    "Medea":["Colleoni"],"Mercurio":["tempio","san","assiso"],"Mirra":["San"],"Minerva":["autoritratto","chiesa","tempio"],
    "Narciso":["san"],"Nereo":["ritratto","san","santi","SS."],"Nettuno":["veduta"],"Oceano":["Atlantico"],
    "Oreste":["Ritratto","autoritratto"],"Pan":["zucchero","veduta"],"Patroclo":["san"],"Pirro":["Stipiciano"],"Pluto":["Topolino","Disney"],
    "Polidoro":["ritratto","Vicolo"],"Polissena":["Ritratto","santa"],"Priamo":["san","santo","Sulis","Fumagalli"],"Scilla":["topografica","veduta","paesaggio"],
    "Telemaco":["stemma","ritratto"],"Teseo":["Spirito"],"Tolomeo":["San","Santi","Ritratto","beato"],"Urano":["Monte"],
    "Venere":["Giorgio","Castello","modella","tempio"],"Verità":["stemma","Pierantonio","ritratto"],"Vittorie":["reggitarga"],
    "Speranza":["ritratto","santa","chiesa","papa","san","santo","madonna"],"Scilla":["ritratto"],"Serapide":["tempio"],"Achille":["stemma","ritratto","duomo","san"],
    "Adone":["sant"],"Cibele":["tempio"],"Cirene":["simone","palazzo"],"Teseo":["tempio","ritratto"],"Titano":["san"],"Ulisse":["ritratto","Dini","cardinalizia"],"Zeus":["tempio"]

}


def validate_matches(row):
    subject = row["subject"]
    valid_matches = []

    for match in row["matched_term"]:

        # PRIMO CHECK: controlla se il termine è nelle exclusion_rules
        if match in exclusion_rules:
            exclusion_keywords = exclusion_rules[match]
            # se almeno una parola di esclusione è presente, scarta il termine
            if any(re.search(r'\b' + re.escape(k) + r'\b', subject, re.IGNORECASE) for k in exclusion_keywords):
                continue  
        
        # SECONDO CHECK: controlla context_rules 
        # se il termine non ha regole → tienilo
        if match not in context_rules:
            valid_matches.append(match)
            continue

        # controlla parole contestuali se il match è ambiguo 
        keywords = context_rules[match]

        if any(re.search(r'\b' + re.escape(k) + r'\b', subject, re.IGNORECASE) for k in keywords):
            valid_matches.append(match)

    return valid_matches


df_arco["matched_term"] = df_arco["subject"].str.findall(regex) #trova tutti i termini e restituisce la lista
df_arco["matched_term"] = df_arco.apply(validate_matches, axis=1) #fa i controlli definiti sopra 

# df che risulta
df_matches = df_arco[df_arco["matched_term"].str.len() > 0]

In [20]:
# Verifica i risultati
print(f"Totale beni Arco: {len(df_arco)}")
print(f"Beni con match mitologico: {len(df_matches)}")

Totale beni Arco: 522846
Beni con match mitologico: 9674


During the manual inspection of the filtered dataset, it emerged that several relevant mythological entities were frequently written in lowercase within cataloguing metadata. This issue affected mythological creatures, collective entities and some recurring subjects whose capitalization was not applied consistently in ArCo descriptions. 

To recover these missing entities without reintroducing excessive semantic noise, a second filtering iteration was performed using a restricted list of selected mythology-related terms.

The second validation step search for new mythology-related occurrences that had not already been identified during the first matching process to avoid duplication. 

In [21]:
#ulteriore iterazione NON case sensitive 
extra_check_terms = [
    "Aracne", "Argo", "Argonauti", "Arpia", "Arpie","Ascanio", "Callisto", "Centauro", "Cerva di Cerinea", "ciclope", "ciclopi", 
    "Chimera", "Cinghiale calidonio", "Cinghiale di Erimanto","Cupido", "Didone", "Elena", "Ippocampo",
    "Ippolito", "idra", "Minotauro", "musa", "muse", "Ratto di Europa","Sibilla samia", "Sibilla eritrea", "Sibilla tiburtina","Sirena", "Siringa", "Tritone", "satiro", "satiri"
]


extra_terms = [re.escape(t) for t in extra_check_terms]  #mette in sicurezza le parole per la regex 

extra_pattern = r'(?<![A-Za-zÀ-ÿ])(' + '|'.join(extra_terms) + r')(?![A-Za-zÀ-ÿ])'  
extra_regex = re.compile(extra_pattern, re.IGNORECASE)  #non case sensitive

#creo una copia del dataframe
df_matches = df_matches.copy()

# converte lista in set per confronto veloce
df_matches["matched_term_set"] = df_matches["matched_term"].apply(
    lambda x: set(x) if isinstance(x, list) else set()
)

#nuovo check
def find_new_matches(row):
    subject = row["subject"]
    already_found = row["matched_term_set"]

    all_hits = set(extra_regex.findall(subject))

    # rimuove quelli già trovati nel primo passaggio
    new_hits = all_hits - already_found

    return list(new_hits)


#applichiamo la funzione e creiamo una nuova colonna con gli extra_matched_terms
df_matches["extra_matched_terms"] = df_matches.apply(find_new_matches, axis=1)

#teniamo solo le righe utili
df_extra_hits = df_matches[df_matches["extra_matched_terms"].str.len() > 0].copy()


<h3>Normalizing and Merging Mythological Matches</h3>
After completing the two filtering phases, the extracted mythology-related matches were further processed in order to normalize and consolidate the identified terms.

At this stage, the dataset contained the original matches retrieved through the first case-sensitive filtering process and the additional matches identified through the second non-case-sensitive iteration. Since the same mythological subject could appear in different lexical forms, a <b>normalization</b> process was necessary to male the comparison beetween terms more consistent.

A second objective of this phase was to <b>preserve the most semantically specific mythology-related terms</b> whenever multiple overlapping terms appeard within the same subject description, for example "Europa" and "Ratto di Europa". The process prioritized the more specific expression whenever a generic term was already contained within a more informative mythology-related reference.
This strategy improved the semantic quality of the dataset by reducing ambiguity and preserving more meaningful mythological references. 

The final validated mythology-related terms were stored in a dedicated column representing the definitive set of mythological subjects associated with each cultural heritage object.


In [22]:
#normalizziamo i termini per poterli confrontare 
def normalize(s):
    return s.lower().strip() if isinstance(s, str) else ""

def merge_terms(row):
    base = row["matched_term"] if isinstance(row["matched_term"], list) else []
    extra = row["extra_matched_terms"] if isinstance(row["extra_matched_terms"], list) else []

    base_norm = {normalize(x): x for x in base}
    extra_norm = {normalize(x): x for x in extra}

    final = set()

    # 1. tengo gli extra che sono più specifici (es. europa --> ratto d'europa)
    for k, v in extra_norm.items():
        final.add(v)

    for b_key, b_val in base_norm.items():
        is_redundant = False

        for e_key in extra_norm:
            if b_key in e_key and b_key != e_key:
                is_redundant = True
                break

        if not is_redundant:
            final.add(b_val)  #tengo i termini base solo se non c'è una versione più specifica

    return list(final)

#applichiamo la funzione per avere i matched_term completi e più specifici
df_matches["final_terms"] = df_matches.apply(merge_terms, axis=1)

#df_matches[["subject", "final_terms"]].head()
df_checked = df_matches.sort_values(by="final_terms")[["item","label","subject", "final_terms"]]


In [23]:
print(len(df_checked))

9674


After completing the filtering, validation, and normalization processes, the final dataset was exported as a CSV file.

In [24]:
#esportiamo i risultati in un file csv

df_checked.to_csv("arco_myth_matches.csv", index=False, encoding="utf-8")


### Integrating ArCo resources with Wikidata matching entities

Next, data extracted from ArCo and filtered according to mythological characters retrieved from Wikidata was integrated with the URIs of the matched Wikidata entities.

In [25]:
# integrazione entities wikidata rappresentate in entities arco

df_wd_merged = df_wikidata.melt(id_vars=["entity"], value_vars=["label", "aliases"])[
    ["entity", "value"]
].dropna()
df_wd_merged["value"] = df_wd_merged["value"].str.strip().str.lower()

# Dictionary: {term: uri}
mapping_dict = dict(zip(df_wd_merged["value"], df_wd_merged["entity"]))
print("The mapping dictionary:", mapping_dict)

def map_strings_to_uris(string_list):
    if not isinstance(string_list, list):
        return None

    # Cerchiamo l'URI nel dizionario per ogni parola (rimuovendo spazi e minuscole)
    uris = []
    for s in string_list:
        clean_string = str(s).strip().lower()
        if clean_string in mapping_dict:
            uris.append(mapping_dict[clean_string])

    if not uris:
        return None
    elif len(uris) == 1:
        return uris[0]  
    else:
        return ", ".join(uris)  # concatenate in one line if there is more than one

    
    
df_arco_wd_entities = df_checked.copy()
print(df_arco_wd_entities.shape)
#print(df_arco_wd_entities["final_terms"].dtype)
df_arco_wd_entities["wd_entities"] = df_arco_wd_entities["final_terms"].apply(map_strings_to_uris) # l'input è a riga del df
print("The updated dataframe:\n", df_arco_wd_entities.head(5))
print("Entity column:\n", df_arco_wd_entities["wd_entities"])

The mapping dictionary: {'abarbarea': 'http://www.wikidata.org/entity/Q279782', 'acanto': 'http://www.wikidata.org/entity/Q9143205', 'acaste': 'http://www.wikidata.org/entity/Q9143220', 'accò': 'http://www.wikidata.org/entity/Q3604219', 'acefalo': 'http://www.wikidata.org/entity/Q417341', 'acheloo': 'http://www.wikidata.org/entity/Q391379', 'acheso': 'http://www.wikidata.org/entity/Q605797', 'achlys': 'http://www.wikidata.org/entity/Q340545', 'acmone': 'http://www.wikidata.org/entity/Q33191300', 'acrato': 'http://www.wikidata.org/entity/Q12647288', 'adefagia': 'http://www.wikidata.org/entity/Q356321', 'adikia': 'http://www.wikidata.org/entity/Q2824393', 'admete': 'http://www.wikidata.org/entity/Q17748154', 'adone': 'http://www.wikidata.org/entity/Q163920', 'adrastea': 'http://www.wikidata.org/entity/Q16737262', 'aergia': 'http://www.wikidata.org/entity/Q3844469', 'afaia': 'http://www.wikidata.org/entity/Q618401', 'afrodite': 'http://www.wikidata.org/entity/Q35500', 'afrodite pandemos':

In [26]:
# dump csv integrato con uris wikidata
df_arco_wd_entities.to_csv("arco_wd_entities.csv", index=False, encoding="utf-8")

<h3>Enriching the Mythology Dataset with Additional ArCo Metadata</h3>

After identifying mythology-related cultural heritage items, the dataset was enriched with additional metadata extracted from the ArCo knowledge graph.

The objective of this phase was to integrate further descriptive information useful for the analytical and visualization stages of the project. 

The enrichment process could not be performed during the initial extraction phase because of technical limitations associated with the ArCo SPARQL endpoint.

Due to the very large number of cultural heritage items and requested properties, complex SPARQL queries frequently produced timeouts or execution failures.

For this reason, the workflow was divided into multiple smaller queries executed separately.

To manage the large amount of requested data, a <b>reusable batch-query function</b> was created.

Instead of querying all mythology-related items simultaneously, the workflow divided the dataset into smaller groups of entities and progressively submitted multiple SPARQL requests to the ArCo endpoint.

In [27]:
arco_endpoint = "https://dati.cultura.gov.it/sparql"
sparql_arco = SPARQLWrapper(arco_endpoint)
sparql_arco.setReturnFormat(JSON)

df_base = pd.read_csv("arco_myth_matches.csv", encoding="utf-8")
items = df_base["item"].dropna().unique().tolist() #estraggo la lista di item unici per poi fare ulteriori query su Wikidata

In [28]:
# funzione per eseguire query in batch su SPARQL

def run_batched_query(items_list, query_template, batch_size=100):

    all_dfs = []

    for i in range(0, len(items_list), batch_size):

        batch = items_list[i:i + batch_size]
        formatted_items = " ".join(f"<{uri}>" for uri in batch) 

        query = query_template.replace("{ITEMS}", formatted_items)

        sparql_arco.setQuery(query)

        try:
            results = sparql_arco.query().convert()

            rows = []
            for r in results["results"]["bindings"]:
                rows.append({k: v["value"] for k, v in r.items()})

            if rows:
                all_dfs.append(pd.DataFrame(rows))

        except Exception as e:
            print(f"Errore batch {i}: {e}")

    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)

    return pd.DataFrame()

In [29]:
#QUERY 1: identifier, creators, types

query_ict= """ 
        PREFIX arco: <https://w3id.org/arco/ontology/arco/>
        PREFIX a-dd: <https://w3id.org/arco/ontology/denotative-description/>
        PREFIX a-loc: <https://w3id.org/arco/ontology/location/>
        PREFIX dc: <http://purl.org/dc/elements/1.1/>
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        
        SELECT ?item ?identifier      
            (GROUP_CONCAT(DISTINCT ?creator; separator=", ") AS ?creators) 
            (SAMPLE (?creatorL) AS ?creatorLabel) 

            (GROUP_CONCAT(DISTINCT ?type; separator=", ") AS ?types) 
            (SAMPLE (?typeL) AS ?typeLabel) 

            (GROUP_CONCAT(DISTINCT ?instituteOrSite; separator=", ") AS ?institutesOrSites)
            (SAMPLE (?instituteOrSiteL) AS ?institutesOrSiteLabel) 
            
            (GROUP_CONCAT(DISTINCT ?place; separator=", ") AS ?coverage)
            (SAMPLE (?placeL) AS ?coverageLabel)
        
        WHERE {
            VALUES ?item {{ITEMS}} .
            ?item arco:uniqueIdentifier ?identifier .
 
            OPTIONAL {
                ?item dc:creator ?creator . 
                OPTIONAL {?creator rdfs:label ?creatorL .}
                }

            OPTIONAL {
                ?item a-dd:hasCulturalPropertyType ?type . 
                OPTIONAL {
                    ?type rdfs:label ?typeL .
                    FILTER(LANG(?typeL) = "it")
                    FILTER (!CONTAINS(STR(?typeL), "Tipo del bene:"))       
                }
            } 

            OPTIONAL {
                ?item a-loc:hasCulturalInstituteOrSite ?instituteOrSite . 
                OPTIONAL {?instituteOrSite rdfs:label ?instituteOrSiteL .}
                }

            OPTIONAL {
                ?item dc:coverage ?place .
                OPTIONAL {?place rdfs:label ?placeL .}
            } 
        }    
        
 
        GROUP BY ?item ?identifier
        """

In [30]:
#Q2 creation - date range 

query_date = """ 
        PREFIX arco: <https://w3id.org/arco/ontology/arco/>
        PREFIX a-cd: <https://w3id.org/arco/ontology/context-description/>

        SELECT ?item 
            (SAMPLE(STR(?event)) AS ?event)
            (SAMPLE(STR(?startDate)) AS ?start) 
            (SAMPLE(STR(?endDate)) AS ?end)
        WHERE {
            VALUES ?item 
                {{ITEMS}}
    
        OPTIONAL {
            ?item a-cd:hasDating ?dating .
            ?dating a-cd:hasDatingEvent ?event .
            ?event a-cd:specificTime ?time .
            ?time arco:startTime ?startDate ;
              arco:endTime ?endDate .
        
            # Filtro per assicurarci di prendere solo la creazione
            FILTER(REGEX(STR(?event), "creation", "i"))
        }        
    }
    GROUP BY ?item 
        """

In [31]:
#QUERY 3 - geometries and locations 
query_location = """ 
        PREFIX a-loc: <https://w3id.org/arco/ontology/location/>
        PREFIX clvapit: <https://w3id.org/italia/onto/CLV/>
        SELECT ?item
            (SAMPLE(STR(?geometry)) AS ?geometry) 
            (SAMPLE(STR(?lat)) AS ?lat)
            (SAMPLE(STR(?long)) AS ?long)
        WHERE {
            VALUES ?item {{ITEMS}} .
            # OPTIONAL per coordinate geografiche
            OPTIONAL {
                ?item clvapit:hasGeometry ?geometry .
                ?geometry a-loc:hasCoordinates ?coordinates .
                ?coordinates a-loc:lat ?lat ;
                            a-loc:long ?long .
            }        
        }
        GROUP BY ?item 
        """

In [32]:
#QUERY 4 - images 
query_img = """ 
        PREFIX foaf: <http://xmlns.com/foaf/0.1/>
 
        SELECT ?item (SAMPLE(?image) AS ?sampleImage) #  solo una tra quelle disponibili
        WHERE {
            VALUES ?item {{ITEMS}} .
            OPTIONAL {?item foaf:depiction ?image . }        
        }
	GROUP BY ?item
        """

Each category of metadata was stored temporarily in an independent DataFrame before the final integration phase.

In [33]:
#eseguo le query 
df_ict = run_batched_query(items, query_ict, batch_size=50)


In [34]:
df_date = run_batched_query(items, query_date,batch_size=50)

In [35]:
df_location = run_batched_query(items, query_location, batch_size=50)

In [36]:
df_img = run_batched_query(items, query_img, batch_size=50)

After completing all SPARQL queries, the retrieved metadata was merged into a single <b>enriched dataset</b> using the cultural heritage item URI as a common identifier.

The final enriched dataset combined:
- the original mythology-related filtering results,
- and all additional metadata retrieved from ArCo.

In [37]:
#creo un df unico 
df_enriched = df_base \
    .merge(df_ict, on="item", how="left") \
    .merge(df_date, on="item", how="left") \
    .merge(df_location, on="item", how="left") \
    .merge(df_img, on="item", how="left")


In [38]:
df_enriched.to_csv("arco_myth_enriched.csv", index=False)
print(len(df_enriched))

9674


In [39]:
#VISUALIZZIAMO SOLO LE LABEL DEL DF 

df = pd.read_csv("arco_myth_enriched.csv")

colonne_selezionate = [
    "item",
    "label",
    "subject",
    "final_terms",
    "typeLabel",
    "creatorLabel",
    "institutesOrSiteLabel",
    "coverage",
    "start",
    "end"
]

df_label = df[colonne_selezionate].copy()
df_label.head(2)

,item,label,subject,final_terms,typeLabel,creatorLabel,institutesOrSiteLabel,coverage,start,end
0,https://w3id.org/arco/resource/HistoricOrArtis...,"allegoria dell'Abbondanza (candeliere, pendant...",allegoria dell'Abbondanza,['Abbondanza'],candeliere,NaN,NaN,Torino (TO),post 1840,ante 1860
1,https://w3id.org/arco/resource/HistoricOrArtis...,Abbondanza (arazzo) - manifattura fiorentina (...,Abbondanza,['Abbondanza'],arazzo,NaN,NaN,Firenze (FI),post 1600,ante 1649


The DataFrame storing basic information about historical-artistic resources and integrated with the corresponding Wikidata mythological entities was then merged with the enriched DataFrame, in order to store all the needed information in one complete DataFrame.

In [42]:
# df arco enriched con uris wikidata

df_arco_wd =  pd.merge(df_enriched, df_arco_wd_entities, on="item", how="left")
#print(f"Entities df:\n {df_arco_wd['wd_entities']}")
df_arco_wd.head(5)
#print(f"The final df has {len(df_arco_wd)} rows")


# find and combine duplicate _x and _y columns
for col in df_enriched.columns:
    if col != "item" and f"{col}_x" in df_arco_wd.columns:
        # Combine the two columns (prefers _x, fills NaNs with _y)
        df_arco_wd[col] = df_arco_wd[f"{col}_x"].combine_first(df_arco_wd[f"{col}_y"])
        
        # Drop the duplicate columns
        df_arco_wd.drop(columns=[f"{col}_x", f"{col}_y"], inplace=True)

df_arco_wd.head(2)
#print(f"The final df has {len(df_arco_wd)} rows")

,item,identifier,creators,types,typeLabel,institutesOrSites,institutesOrSiteLabel,coverage,creatorLabel,event,start,end,geometry,lat,long,sampleImage,wd_entities,label,subject,final_terms
0,https://w3id.org/arco/resource/HistoricOrArtis...,0100200267,,https://w3id.org/arco/resource/CulturalPropert...,candeliere,,NaN,Torino (TO),NaN,https://w3id.org/arco/resource/Event/010020026...,post 1840,ante 1860,https://w3id.org/arco/resource/Geometry/010020...,45.074534,7.678605,https://www.sigecweb.beniculturali.it/images/f...,http://www.wikidata.org/entity/Q336003,"allegoria dell'Abbondanza (candeliere, pendant...",allegoria dell'Abbondanza,['Abbondanza']
1,https://w3id.org/arco/resource/HistoricOrArtis...,0900745503,,https://w3id.org/arco/resource/CulturalPropert...,arazzo,,NaN,Firenze (FI),NaN,https://w3id.org/arco/resource/Event/090074550...,post 1600,ante 1649,https://w3id.org/arco/resource/Geometry/090074...,43.817586,11.233752,http://www.sigecweb.beniculturali.it/images/fu...,http://www.wikidata.org/entity/Q336003,Abbondanza (arazzo) - manifattura fiorentina (...,Abbondanza,['Abbondanza']


One last necessary step in cleaning data extracted from ArCo was related to dates.

The data model used by ArCo allows the start and end dates associated with an event to be expressed at various levels of granularity (century, year, year-month-day) and certainty (year range, *ante*/*post*, circa).

For this reason, dates were processed to enable subsequent temporal data visualization. Specifically, textual descriptions and symbols were removed from uncertain dates and intervals; for *ante* and *post* dates, a year was subtracted and added, respectively.

In order to preserve the data, columns with the original chronological information were maintained. 

In [44]:
# extend df with normalized dates
def clean_dates(string_date):
    date_to_lower = str(string_date).lower().strip()
    match = date_to_lower
    
    if not match:
        return pd.Series({
            'norm_date': None
        })
    
    def process(block):
        block = block.strip()
        
        # Colonna normalizzata (gestione dei vari formati)
        norm = None
        
        # formato date (yyyy/mm/dd)
        date_format = re.search(r'(\d{4})[/-](\d{2})[/-](\d{2})', block)
        
        # intervallo anno-anno (es. 1881-1882)
        interval = re.search(r'(\d{4})-(\d{4})', block)

        # anno
        year_match = re.search(r'\d{4}', block)

        if date_format:
            # Estrarre gruppi e sostituire / con -
            norm = f"{date_format.group(1)}-{date_format.group(2)}-{date_format.group(3)}"
        
        elif interval:
            # estrarre margine estremo dell'intervallo
            norm = f"{interval.group(2)}"

        elif year_match:
            year_val = int(year_match.group(0))
            # aggiungere un anno per 'post'
            if 'post' in block:
                norm = year_val + 1
            # sottrarre un anno per 'ante'
            elif 'ante' in block:
                norm = year_val - 1
            else:
                norm = year_val
        
        return norm

    # restituire data normalizzata
    norm_date = process(match)
    
    return pd.Series({
        'norm_date': norm_date,
    })

In [45]:
# apply function
# column for normalized start date
df_arco_wd[['start_norm']] = df_arco_wd['start'].apply(clean_dates)
# column for normalized end date
df_arco_wd[['end_norm']] = df_arco_wd['end'].apply(clean_dates)

df_arco_wd[['start', 'start_norm', 'end', 'end_norm']].head(20)

,start,start_norm,end,end_norm
0,post 1840,1841,ante 1860,1859
1,post 1600,1601,ante 1649,1648
2,1700,1700,1724,1724
3,1550,1550,1599,1599
4,ca 1730,1730,ca 1732,1732
5,1450,1450,1499,1499
6,ca 1675,1675,ca 1699,1699
7,1575,1575,1599,1599
8,1790,1790,1799,1799
9,post 1909,1910,ante 1909,1908


In [47]:
# order complete dataframe columns
# Ordine alfabetico (A-Z)
df_arco_wd = df_arco_wd[["item", "label", "subject", "final_terms", "wd_entities", "identifier", "creators", "creatorLabel", "types", "typeLabel", "institutesOrSites", "institutesOrSiteLabel", "coverage", "event", "start", "end", "start_norm", "end_norm", "geometry", "lat", "long", "sampleImage"]]

df_arco_wd

,item,label,subject,final_terms,wd_entities,identifier,creators,creatorLabel,types,typeLabel,...,coverage,event,start,end,start_norm,end_norm,geometry,lat,long,sampleImage
0,https://w3id.org/arco/resource/HistoricOrArtis...,"allegoria dell'Abbondanza (candeliere, pendant...",allegoria dell'Abbondanza,['Abbondanza'],http://www.wikidata.org/entity/Q336003,0100200267,,NaN,https://w3id.org/arco/resource/CulturalPropert...,candeliere,...,Torino (TO),https://w3id.org/arco/resource/Event/010020026...,post 1840,ante 1860,1841,1859,https://w3id.org/arco/resource/Geometry/010020...,45.074534,7.678605,https://www.sigecweb.beniculturali.it/images/f...
1,https://w3id.org/arco/resource/HistoricOrArtis...,Abbondanza (arazzo) - manifattura fiorentina (...,Abbondanza,['Abbondanza'],http://www.wikidata.org/entity/Q336003,0900745503,,NaN,https://w3id.org/arco/resource/CulturalPropert...,arazzo,...,Firenze (FI),https://w3id.org/arco/resource/Event/090074550...,post 1600,ante 1649,1601,1648,https://w3id.org/arco/resource/Geometry/090074...,43.817586,11.233752,http://www.sigecweb.beniculturali.it/images/fu...
2,https://w3id.org/arco/resource/HistoricOrArtis...,L'Abbondanza (scultura) - ambito bellunese (pr...,L'Abbondanza,['Abbondanza'],http://www.wikidata.org/entity/Q336003,0500051469,,NaN,https://w3id.org/arco/resource/CulturalPropert...,scultura,...,Belluno (BL),https://w3id.org/arco/resource/Event/050005146...,1700,1724,1700,1724,NaN,NaN,NaN,https://sigecweb.beniculturali.it/images/fulls...
3,https://w3id.org/arco/resource/HistoricOrArtis...,Abbondanza (dipinto) - ambito romano (seconda ...,Abbondanza,['Abbondanza'],http://www.wikidata.org/entity/Q336003,1200237830,,NaN,https://w3id.org/arco/resource/CulturalPropert...,dipinto,...,Bolsena (VT),https://w3id.org/arco/resource/Event/120023783...,1550,1599,1550,1599,NaN,NaN,NaN,http://www.sigecweb.beniculturali.it/images/fu...
4,https://w3id.org/arco/resource/HistoricOrArtis...,Abbondanza e Misericordia (dipinto) by Lama Gi...,Abbondanza e Misericordia,['Abbondanza'],http://www.wikidata.org/entity/Q336003,1500297849,https://w3id.org/arco/resource/Agent/f2fdeea6b...,Lama Giovan Battista - 1673 ca./ 1748,https://w3id.org/arco/resource/CulturalPropert...,dipinto,...,Napoli (NA),https://w3id.org/arco/resource/Event/150029784...,ca 1730,ca 1732,1730,1732,https://w3id.org/arco/resource/Geometry/150029...,40.856624,14.239285,https://sigecweb.beniculturali.it/images/fulls...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9669,https://w3id.org/arco/resource/HistoricOrArtis...,"rilievo con figura muliebre e fanciullo, figur...","rilievo con figura muliebre e fanciullo, figur...","['satiro', 'Apollo']","https://www.wikidata.org/wiki/Q163709, http://...",1500394548,https://w3id.org/arco/resource/Agent/361552d66...,Cataneo Aniello (1732 Ca./ Post 1805),https://w3id.org/arco/resource/CulturalPropert...,stampa,...,Napoli (NA),https://w3id.org/arco/resource/Event/150039454...,post 1775,ANTE 1799,1776,1798,NaN,NaN,NaN,https://sigecweb.beniculturali.it/images/fulls...
9670,https://w3id.org/arco/resource/HistoricOrArtis...,"rilievo con figura muliebre e fanciullo, figur...","rilievo con figura muliebre e fanciullo, figur...","['satiro', 'Apollo']","https://www.wikidata.org/wiki/Q163709, http://...",1500394135,https://w3id.org/arco/resource/Agent/6a0fa2e88...,Aloja Raffaele (notizie 1735-1805),https://w3id.org/arco/resource/CulturalPropert...,stampa controfondata,...,Napoli (NA),https://w3id.org/arco/resource/Event/150039413...,post 1775,ANTE 1799,1776,1798,NaN,NaN,NaN,https://sigecweb.beniculturali.it/images/fulls...
9671,https://w3id.org/arco/resource/HistoricOrArtis...,sibilla Samia e sibilla Eritrea (dipinto) - am...,sibilla Samia e sibilla Eritrea,"['sibilla Eritrea', 'sibilla Samia']","http://www.wikidata.org/entity/Q1121944, http:...",1300059357,,NaN,https://w3id.org/arco/resource/CulturalPropert...,dipinto,...,Alanno (PE),https://w3id.org/arco/resource/Event/130005935..

In [49]:
# save complete dataframe as csv
df_arco_wd.to_csv("arco_wd_final.csv", encoding="utf-8", index=False)

### Extending Wikidata entities matched by ArCo resources

Finally, the dataset was enriched by extracting additional properties for the Wikidata entities that were matched by ArCo resources.

eccetera

In [33]:
# ADESSO PROVIAMO A FARE ENTITY ENRICHMENT CON WIKIDATA ! ! ! ! 

In [34]:
#dividiamo i final_terms
df_terms = df_matches.explode("final_terms")


In [35]:
#creiamo un vocabolario con le entità trovate
unique_terms = (
    df_terms["final_terms"]
    .dropna()
    .str.lower()
    .unique()
    
)

terms_set = set(unique_terms)
print(f"soggetti unici normalizzati: {len(terms_set)}")

soggetti unici normalizzati: 468


In [36]:
print(terms_set)

{'leandro', 'clio', 'eurinome', 'dionisio', 'perseo', 'mnemosine', 'procri', 'andromaca', 'notte', 'atteone', 'leda', 'olimpo', 'esone', 'creusa', 'eurialo', 'oto', 'penteo', 'poseidone', 'ifigenia', 'dafne', 'ganimede', 'telemaco', 'iride', 'saturno', 'euridice', 'selene', 'roma', 'partenope', 'febbre', 'nefele', 'euristeo', 'zete', 'vertumnus', 'galeso', 'fortuna', 'cleobi', 'helios', 'satiro', 'deianira', 'sibilla samia', 'aretusa', 'penelope', 'urano', 'epimeteo', 'titone', 'eridano', 'cinghiale calidonio', 'hypnos', 'caos', 'anassagora', 'glauco', 'anfione', 'apsirto', 'pelope', 'tesi', 'antinoe', 'endimione', 'andromeda', 'atena', 'polluce', 'anfitrite', 'tre grazie', 'nesso', 'orfeo', 'danae', 'alcione', 'giove tonante', 'mater matuta', 'calipso', 'eubuleo', 'metide', 'aglaia', 'apollo', 'caco', 'sirena', 'eracle', 'aracne', 'asclepio', 'argonauti', 'circe', 'atalanta', 'mentore', 'onfale', 'vulcano', 'ratto di europa', 'venere', 'trittolemo', 'phanes', 'eros', 'io', 'almone', '

In [37]:
#dobbiamo recuperare gli uri dal df di wikidata 
#prima dobbiamo splittare gli aliases 

def alias_match(alias_string, terms_set):

    if pd.isna(alias_string):
        return False

    aliases = [a.strip().lower() for a in alias_string.split(",")] #divido alias separati da ,

    return any(alias in terms_set for alias in aliases)

In [38]:
df_wikidata_filtered = df_wikidata[
    (
        # match diretto sulla label
        df_wikidata["label"]
        .str.strip()
        .str.lower()
        .isin(terms_set)
    )
    |
    (
        # match sugli aliases
        df_wikidata["aliases"].apply(
            lambda x: alias_match(x, terms_set)
        )
    )
].copy()


print(f"soggetti wikidata che matchano i termini unici: {len(df_wikidata_filtered)}")

df_wikidata_filtered[["entity", "label", "aliases"]].head()

#sono meno perchè aliases e label sono la stessa entity, quindi Nike e Vittoria Alata = 1 soggetto wikidata 
#sono meno perchè 'ciclope','ciclopi','idra','musa','muse','putti','putto','satiri','satiro' li ho messi io a mano nel secondo check sui soggetti. Non ci sono nel csv di wikidata 

soggetti wikidata che matchano i termini unici: 430


,entity,label,aliases
5,http://www.wikidata.org/entity/Q391379,Acheloo,NaN
14,http://www.wikidata.org/entity/Q163920,Adone,Atunis
19,http://www.wikidata.org/entity/Q35500,Afrodite,"Citerèa, Citerea, Aφροδίτη, Cipride"
26,http://www.wikidata.org/entity/Q4439972,Aglaia,NaN
35,http://www.wikidata.org/entity/Q911084,Alfeo,NaN


In [39]:
#CONTROLLO PER VEDERE SE TUTTO TORNA 

wikidata_labels = set(
    df_wikidata_filtered["label"]
    .dropna()
    .str.strip()
    .str.lower()
)


wikidata_aliases = set()

for aliases in df_wikidata_filtered["aliases"].dropna():

    split_aliases = [
        a.strip().lower()
        for a in aliases.split(",")
    ]

    wikidata_aliases.update(split_aliases)

wikidata_terms = wikidata_labels.union(wikidata_aliases)

matched_terms = terms_set.intersection(wikidata_terms)
missing_terms = terms_set - wikidata_terms

print("TERMINI ARCO TOTALI:", len(terms_set))
print("TERMINI MATCHATI:", len(matched_terms))
print("TERMINI MANCANTI:", len(missing_terms))

missing_terms_sorted = sorted(missing_terms)

TERMINI ARCO TOTALI: 468
TERMINI MATCHATI: 468
TERMINI MANCANTI: 0


In [40]:
#proprietà che vogliamo estrarre (piccolo test per ora)
#instance of (P31)
#sex or gender (P21)
#domain of saint or deity (P2925)
#said to be the same as (P460)


In [41]:
# PROVIAMO A ESTRARRE LE MALEDETTE PROPRIETA CON QUESTO METODO PER NON FARCI BANNARE DA WIKIDATA

#iniziamo estraendo i QID delle entità che ci interessano 
df_wikidata_filtered = df_wikidata_filtered.copy()
df_wikidata_filtered["qid"] = (
    df_wikidata_filtered["entity"]
    .str.split("/")
    .str[-1]
)

#mettiamoli i una lista 

qids = df_wikidata_filtered["qid"].dropna().unique().tolist()

print(len(qids))

430


In [42]:
headers = {
    "User-Agent": "ArCo-MythProject/1.0 (contact: anna.pasetto2@studio.unibo.it)"
}

In [43]:
label_cache = {}

def get_label(qid):

    if qid is None:
        return None

    if qid in label_cache:
        return label_cache[qid]

    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"

    try:
        r = requests.get(url, headers=headers, timeout=10)
        data = r.json()

        label = (
            data["entities"][qid]["labels"]
            .get("en", {})
            .get("value")
        )

    except:
        label = None

    label_cache[qid] = label
    return label

In [44]:
#selezioniamo solo gli instance_of che ci interessano per poterli estrarre

preferred_p31 = {
    "Q11688446",
    "Q22988604",
    "Q111252729",
    "Q13405593",
    "Q24336068",
    "Q22989102",
    "Q20902363",
    "Q124940323",
    "Q24334685"
}

In [45]:
#visto che non possiamo fare SPARQL Queries, prendiamo direttamente tutta la scheda wikidata di ogni entità (ho stra paura)

def get_wikidata_properties(qid):

    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"

    try:
        r = requests.get(url, headers=headers, timeout=(5, 15))
        r.raise_for_status()
        data = r.json()

    except Exception as e:
        print("Errore su", qid, e)
        return {
            "qid": qid,
            "P31": None,
            "P21": None,
            "P2925": None,
            "P460": None,
            "P1889": None,
        }

    entity = data["entities"][qid]
    claims = entity.get("claims", {})

    #funzione generica per prendere il primo valore 
    def extract(prop):

        if prop not in claims:
            return None

        try:
            value = claims[prop][0]["mainsnak"]["datavalue"]["value"]

            if isinstance(value, dict) and "id" in value:
                return value["id"]

            return value

        except:
            return None

    #funzione separata per p31

    def extract_p31():

        if "P31" not in claims:
            return None

        found = []

        for claim in claims["P31"]:

            try:

                value = claim["mainsnak"]["datavalue"]["value"]

                if isinstance(value, dict) and "id" in value:

                    qid_value = value["id"]

                    found.append(qid_value)

                    # se troviamo uno dei tipi preferiti
                    if qid_value in preferred_p31:
                        return qid_value

            except:
                continue

        if found:
            return found[0]

        return None

    return {
        "qid": qid,
        "P31": extract_p31(),
        "P21": extract("P21"),
        "P2925": extract("P2925"),
        "P460": extract("P460"),
        "P1889": extract("P1889"), # different from
    }


In [46]:
results = []

for i, qid in enumerate(tqdm(qids)):

    results.append(get_wikidata_properties(qid))

    time.sleep(1)

    if i % 50 == 0 and i != 0:
        pd.DataFrame(results).to_csv("backup_wikidata.csv", index=False)
        print(f"Checkpoint salvato: {i}")

  0%|          | 0/430 [00:00<?, ?it/s]

 12%|█▏        | 51/430 [02:15<14:58,  2.37s/it]

Checkpoint salvato: 50


 23%|██▎       | 101/430 [04:31<15:58,  2.91s/it]

Checkpoint salvato: 100


...

 82%|████████▏ | 351/430 [16:38<03:53,  2.95s/it]

Checkpoint salvato: 350


 93%|█████████▎| 401/430 [19:06<01:22,  2.85s/it]

Checkpoint salvato: 400


100%|██████████| 430/430 [20:26<00:00,  2.85s/it]


In [47]:
df_props = pd.DataFrame(results)
df_props.head(10)

,qid,P31,P21,P2925,P460,P1889
0,Q391379,Q3027575,Q6581097,Q203923,None,None
1,Q163920,Q22989102,Q6581097,Q7242,None,None
2,Q35500,Q22989102,Q6581072,Q316,Q47652,Q9141765
3,Q4439972,Q22989102,Q6581072,None,None,None
4,Q911084,Q3027575,Q6581097,Q941745,None,None
5,Q107785,Q182406,Q6581072,None,None,Q108904
6,Q3575327,Q22989102,Q6581097,None,None,None
7,Q180222,Q22989102,Q6581072,Q165,Q2987780,Q186663
8,Q572133,Q22989102,Q6581097,Q316,None,None
9,Q37340,Q22989102,Q6581097,None,Q900649,None


In [48]:
cols = ["P31","P21","P2925","P460", "P1889"] 

def decorate(qid):

    if qid is None:
        return None

    label = get_label(qid)

    if label:
        return f"{qid} ({label})"
    return qid

for col in cols:
    df_props[col] = df_props[col].apply(decorate)



In [49]:
df_props.head()

,qid,P31,P21,P2925,P460,P1889
0,Q391379,Q3027575 (Potamoi),Q6581097 (male),Q203923 (Achelous River),None,None
1,Q163920,Q22989102 (Greek deity),Q6581097 (male),Q7242 (beauty),None,None
2,Q35500,Q22989102 (Greek deity),Q6581072 (female),Q316 (love),Q47652 (Venus),Q9141765 (Afrodyta)
3,Q4439972,Q22989102 (Greek deity),Q6581072 (female),None,None,None
4,Q911084,Q3027575 (Potamoi),Q6581097 (male),Q941745 (Alfeios),None,None


In [50]:
#carichiamo tutto in un df 

df_props.to_csv("entities_and_properties.csv", index=False)

In [51]:
# estrazione delle proprietà con molti oggetti:
# wdt:P451 # unmarried partner
# wdt:P40 # child
# wdt:P26 #spouse
# wdt:P3373 #sibling
#father (P22)
#mother (P25)
#killed by (P157)

# def ...


In [52]:
valid_qids = set(df_wikidata_filtered["qid"])

network_properties = {
    "P22": "father",
    "P25": "mother",
    "P40": "child",
    "P3373": "sibling",
    "P26": "spouse",
    "P451" : "unmarried partner",
    "P157" : "killed_by"
}

In [53]:
#ORA PROVIAMO AD ESTRARRE I NETWORK 
def extract_relations(qid):

    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"

    try:
        r = requests.get(url, headers=headers, timeout=(5,15))
        r.raise_for_status()
        data = r.json()

    except Exception as e:
        print("Errore:", qid, e)
        return []

    entity = data["entities"][qid]
    claims = entity.get("claims", {})

    rows = []

    for prop, relation_name in network_properties.items():

        if prop not in claims:
            continue

        for claim in claims[prop]:

            try:
                value = claim["mainsnak"]["datavalue"]["value"]

                if isinstance(value, dict) and "id" in value:

                    target_qid = value["id"]

                    # tieni SOLO relazioni interne al corpus
                    if target_qid in valid_qids:

                        rows.append({
                            "source": qid,
                            "relation": relation_name,
                            "target": target_qid,
                            "target_label": get_label(target_qid)
                        })

            except:
                continue

    return rows

In [55]:
all_relations = []

for qid in tqdm(qids):

    rels = extract_relations(qid)

    all_relations.extend(rels)

    time.sleep(1)

df_relations = pd.DataFrame(all_relations)

100%|██████████| 430/430 [18:25<00:00,  2.57s/it]


In [56]:
qid_to_label = dict(
    zip(
        df_wikidata_filtered["qid"],
        df_wikidata_filtered["label"]
    )
)

In [57]:
df_relations.to_csv("mythological_network.csv", index=False)

## Representing and refining data

The following sections of the notebook outline the research questions the project aims to answer and the data visualization and storytelling strategies adopted. 

#### Overview: how are mythological artworks distributed across cultural institutions?

#### What are the most represented mythological subjects over time?

In [1]:
#arco myth enriched with dates

#### Are there chronological periods which discover mythology more than others?

In [2]:
#arco myth enriched with dates + groupings

#### Which relations among mythological characters can be extracted starting from ArCo artworks?

#### Female characters are as represented as male characters?

In [3]:
# bar or donut from arco myth enriched + wikidata

#### Who are the most represented female characters?

In [4]:
# bar or donut from arco myth enriched + wikidata

#### Female characters are more represented alone or in association with male characters? This representation changes across time?

In [5]:
#processing of final terms + wikidata + dates

#### Which are the most represented creatures and monsters?

In [7]:
#arco + wikidata (per estrarre category)

#### Creatures and monsters are more or less represented than deities and heroes?

In [6]:
#arco + wikidata (per estrarre category)

#### Are there chronological periods in which creatures and monsters are more relevant than antropomorphic subjects?

In [10]:
#arco con date normalizzate + wikidata (per estrarre category)